# LexIA — Phase 2 : Chunking

Ce notebook retrace la stratégie de chunking du corpus juridique LexIA.
Il couvre :
- Le rappel de la distribution du corpus
- Le choix et la justification des paramètres de chunking
- La comparaison des stratégies (fixed-size vs recursive vs parent-child)
- L'analyse de la distribution des chunks produits
- Les décisions techniques clés

**Input** : 13 644 articles juridiques (`corpus.jsonl`)  
**Output** : chunks prêts pour l'embedding (`chunks.jsonl`)

## 1. Rappel : pourquoi le chunking est critique dans un RAG

Le chunking est la décision qui impacte le plus les performances d'un RAG :
- **Trop grand** : l'embedding capture mal le sens, le retrieval est imprécis
- **Trop petit** : on perd le contexte juridique, la réponse manque de cohérence
- **Sans overlap** : on coupe des phrases au milieu, on perd les transitions

### Stratégies disponibles

| Stratégie | Description | Avantage | Inconvénient |
|---|---|---|---|
| Fixed-size | Coupe tous les N tokens | Simple, rapide | Coupe les phrases |
| Recursive | Coupe sur `\n\n`, `\n`, `.`... | Respecte la structure | Chunks de taille variable |
| Semantic | Coupe sur les changements de sens | Très précis | Lent, coûteux |
| Parent-child | Petit chunk retrieval + grand chunk LLM | Précision + contexte | Plus complexe |

**Choix** : `RecursiveCharacterTextSplitter` adapté à la longueur de chaque article — c'est la stratégie qui offre le meilleur compromis pour un corpus juridique structuré.

## 2. Imports et configuration

In [ ]:
import json
import re
from pathlib import Path
from collections import Counter
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print('Imports OK')

In [ ]:
CORPUS_PATH = '/workspaces/lexia/data/processed/corpus.jsonl'
CHUNKS_PATH = '/workspaces/lexia/data/processed/chunks.jsonl'

# Seuils de catégorisation
SHORT_THRESHOLD = 400    # chars — pas de découpage
LONG_THRESHOLD  = 10000  # chars — chunking agressif

# Paramètres de chunking
# chunk_size=512 : bon compromis précision/contexte pour un modèle d'embedding
# chunk_overlap=64 : ~12% overlap pour ne pas perdre les transitions
CHUNK_SIZE_STANDARD = 512
CHUNK_OVERLAP_STANDARD = 64
CHUNK_SIZE_LONG = 256
CHUNK_OVERLAP_LONG = 32

print('Configuration OK')
print(f'Seuil court  : < {SHORT_THRESHOLD} chars → 1 chunk = 1 article')
print(f'Seuil long   : > {LONG_THRESHOLD} chars → chunking agressif')
print(f'chunk_size standard : {CHUNK_SIZE_STANDARD} chars')
print(f'chunk_size long     : {CHUNK_SIZE_LONG} chars')

## 3. Chargement du corpus

In [ ]:
docs = []
with open(CORPUS_PATH, encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)
        docs.append(Document(
            page_content=record['page_content'],
            metadata=record['metadata'],
        ))

lengths = sorted([len(d.page_content) for d in docs])
n = len(lengths)

print(f'Corpus chargé : {n} articles')
print(f'Longueur médiane : {lengths[n//2]} chars')
print(f'Longueur moyenne : {sum(lengths)//n} chars')
print(f'P90              : {lengths[int(n*0.90)]} chars')

## 4. Analyse de la distribution — choix des paramètres

La distribution guide directement nos choix de `chunk_size` :
- **60% des articles < 500 chars** → ne pas les découper, ils sont déjà de taille chunk
- **P90 = 1 170 chars** → un `chunk_size=512` produit 2-3 chunks max pour 90% des articles
- **16 articles > 10 000 chars** → annexes techniques, chunking plus agressif

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution du corpus — base du choix des paramètres de chunking', fontsize=13)

# Histogramme avec les seuils
ax1 = axes[0]
lengths_clipped = [min(l, 5000) for l in lengths]
ax1.hist(lengths_clipped, bins=60, color='#9FE1CB', edgecolor='white', linewidth=0.5)
ax1.axvline(x=SHORT_THRESHOLD, color='#7F77DD', linestyle='--', linewidth=2,
            label=f'seuil court ({SHORT_THRESHOLD} chars)')
ax1.axvline(x=CHUNK_SIZE_STANDARD, color='#D85A30', linestyle='--', linewidth=2,
            label=f'chunk_size ({CHUNK_SIZE_STANDARD} chars)')
ax1.axvline(x=lengths[n//2], color='#1D9E75', linestyle='-', linewidth=2,
            label=f'médiane ({lengths[n//2]} chars)')
ax1.set_xlabel('Longueur article (chars) — tronqué à 5 000')
ax1.set_ylabel('Nombre articles')
ax1.set_title('Distribution avec seuils de chunking')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# Catégories
ax2 = axes[1]
categories = {
    f'Court\n< {SHORT_THRESHOLD} chars\n1 chunk = 1 article': sum(1 for l in lengths if l < SHORT_THRESHOLD),
    f'Moyen\n{SHORT_THRESHOLD}–{LONG_THRESHOLD} chars\nchunk_size={CHUNK_SIZE_STANDARD}': sum(1 for l in lengths if SHORT_THRESHOLD <= l < LONG_THRESHOLD),
    f'Long (annexes)\n> {LONG_THRESHOLD} chars\nchunk_size={CHUNK_SIZE_LONG}': sum(1 for l in lengths if l >= LONG_THRESHOLD),
}
colors_cat = ['#9FE1CB', '#1D9E75', '#085041']
bars = ax2.bar(range(len(categories)), list(categories.values()),
               color=colors_cat, edgecolor='white')
ax2.set_xticks(range(len(categories)))
ax2.set_xticklabels(list(categories.keys()), fontsize=9)
for bar, count in zip(bars, categories.values()):
    pct = count * 100 // n
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             f'{count:,}\n({pct}%)', ha='center', va='bottom', fontsize=9)
ax2.set_ylabel('Nombre articles')
ax2.set_title('Catégories de chunking')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/workspaces/lexia/notebooks/chunking_categories.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Comparaison des stratégies de chunking

Avant de choisir `RecursiveCharacterTextSplitter`, comparons les approches sur un article exemple.

In [ ]:
# Prend un article moyen pour la démonstration
sample = next(d for d in docs
              if 800 < len(d.page_content) < 1200
              and d.metadata['code_name'] == 'Code du travail')

print(f'Article de démonstration : {sample.metadata["article_num"]}')
print(f'Longueur : {len(sample.page_content)} chars')
print()

# Stratégie 1 : Fixed-size (CharacterTextSplitter)
from langchain_text_splitters import CharacterTextSplitter
splitter_fixed = CharacterTextSplitter(
    chunk_size=512, chunk_overlap=64, separator=' '
)
chunks_fixed = splitter_fixed.split_text(sample.page_content)

# Stratégie 2 : Recursive (notre choix)
splitter_recursive = RecursiveCharacterTextSplitter(
    chunk_size=512, chunk_overlap=64,
    separators=['\n\n', '\n', '.', ';', ',', ' ']
)
chunks_recursive = splitter_recursive.split_text(sample.page_content)

print('── Fixed-size chunking ──────────────────────────')
print(f'Nombre de chunks : {len(chunks_fixed)}')
print(f'Tailles : {[len(c) for c in chunks_fixed]}')
print(f'Fin chunk 1 : ...{chunks_fixed[0][-80:]!r}')
print()
print('── Recursive chunking (notre choix) ────────────')
print(f'Nombre de chunks : {len(chunks_recursive)}')
print(f'Tailles : {[len(c) for c in chunks_recursive]}')
print(f'Fin chunk 1 : ...{chunks_recursive[0][-80:]!r}')

## 6. Définition des splitters et de la fonction de chunking

In [ ]:
splitter_standard = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE_STANDARD,
    chunk_overlap=CHUNK_OVERLAP_STANDARD,
    separators=['\n\n', '\n', '.', ';', ',', ' '],
    length_function=len,
)

splitter_long = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE_LONG,
    chunk_overlap=CHUNK_OVERLAP_LONG,
    separators=['\n\n', '\n', '.', ';', ',', ' '],
    length_function=len,
)

print('Splitters initialisés')
print(f'  Standard : chunk_size={CHUNK_SIZE_STANDARD}, overlap={CHUNK_OVERLAP_STANDARD}')
print(f'  Long     : chunk_size={CHUNK_SIZE_LONG}, overlap={CHUNK_OVERLAP_LONG}')

In [ ]:
def chunk_document(doc):
    """
    Découpe un Document en chunks selon sa longueur.

    Chaque chunk hérite des métadonnées du parent +
    chunk_id, chunk_index, chunk_total, chunk_type, parent_content.

    Point entretien : parent_content permet le parent-child retrieval —
    on retrouve le petit chunk précis mais on envoie l'article complet au LLM.
    """
    content = doc.page_content
    length  = len(content)
    meta    = doc.metadata.copy()

    # Cas 1 : article court — pas de découpage
    if length < SHORT_THRESHOLD:
        meta.update({
            'chunk_id':      f"{meta['article_id']}_0",
            'chunk_index':   0,
            'chunk_total':   1,
            'chunk_type':    'short',
            'parent_content': content,
        })
        return [Document(page_content=content, metadata=meta)]

    # Cas 2 : annexe longue — chunking agressif
    if length > LONG_THRESHOLD:
        splitter   = splitter_long
        chunk_type = 'annexe'
    # Cas 3 : article moyen — chunking standard
    else:
        splitter   = splitter_standard
        chunk_type = 'standard'

    chunks = splitter.split_text(content)
    documents = []
    for i, chunk in enumerate(chunks):
        chunk_meta = meta.copy()
        chunk_meta.update({
            'chunk_id':       f"{meta['article_id']}_{i}",
            'chunk_index':    i,
            'chunk_total':    len(chunks),
            'chunk_type':     chunk_type,
            'parent_content': content,
        })
        documents.append(Document(page_content=chunk, metadata=chunk_meta))

    return documents

print('Fonction chunk_document définie')

## 7. Application du chunking sur le corpus complet

In [ ]:
from tqdm.notebook import tqdm

all_chunks = []
stats = {'short': 0, 'standard': 0, 'annexe': 0}

for doc in tqdm(docs, desc='Chunking'):
    chunks = chunk_document(doc)
    all_chunks.extend(chunks)
    stats[chunks[0].metadata['chunk_type']] += len(chunks)

chunk_lens = [len(c.page_content) for c in all_chunks]

print(f'\n── Statistiques chunks ──────────────────────────')
print(f'Articles en entrée : {len(docs)}')
print(f'Chunks en sortie   : {len(all_chunks)}')
print(f'Ratio expansion    : {len(all_chunks)/len(docs):.2f}x')
print(f'Longueur moyenne   : {sum(chunk_lens)//len(chunk_lens)} chars')
print(f'Longueur max       : {max(chunk_lens)} chars')
print()
print(f'Chunks courts  (short)    : {stats["short"]}')
print(f'Chunks standard           : {stats["standard"]}')
print(f'Chunks annexes            : {stats["annexe"]}')

## 8. Analyse de la distribution des chunks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution des chunks produits', fontsize=13)

# Histogramme chunks vs articles
ax1 = axes[0]
ax1.hist([min(l, 1500) for l in lengths], bins=50,
         alpha=0.6, color='#AFA9EC', label='Articles originaux', edgecolor='white')
ax1.hist([min(l, 1500) for l in chunk_lens], bins=50,
         alpha=0.6, color='#1D9E75', label='Chunks produits', edgecolor='white')
ax1.axvline(x=CHUNK_SIZE_STANDARD, color='#D85A30', linestyle='--',
            linewidth=2, label=f'chunk_size={CHUNK_SIZE_STANDARD}')
ax1.set_xlabel('Longueur (chars) — tronqué à 1 500')
ax1.set_ylabel('Nombre')
ax1.set_title('Articles vs chunks')
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# Répartition par type de chunk
ax2 = axes[1]
labels = ['Courts\n(1 chunk/article)', 'Standard\n(découpés)', 'Annexes\n(découpés agressif)']
values = [stats['short'], stats['standard'], stats['annexe']]
colors_types = ['#9FE1CB', '#1D9E75', '#085041']
wedges, texts, autotexts = ax2.pie(
    values, labels=labels, colors=colors_types,
    autopct='%1.1f%%', startangle=90,
    textprops={'fontsize': 9},
)
ax2.set_title('Répartition par type de chunk')

plt.tight_layout()
plt.savefig('/workspaces/lexia/notebooks/chunks_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Inspection des chunks produits

In [ ]:
# Exemple d'un article découpé en plusieurs chunks
multi = next(
    c for c in all_chunks
    if c.metadata['chunk_type'] == 'standard'
    and c.metadata['chunk_total'] > 2
    and c.metadata['chunk_index'] == 0
)

article_id = multi.metadata['article_id']
article_chunks = [c for c in all_chunks if c.metadata['article_id'] == article_id]

print(f'Article : {multi.metadata["article_num"]}')
print(f'Code    : {multi.metadata["code_name"]}')
print(f'Nombre de chunks : {multi.metadata["chunk_total"]}')
print(f'Longueur article : {len(multi.metadata["parent_content"])} chars')
print()
for i, chunk in enumerate(article_chunks):
    print(f'── Chunk {i+1}/{multi.metadata["chunk_total"]} ({len(chunk.page_content)} chars) ──')
    print(chunk.page_content[:150] + '...')
    print()

In [ ]:
# Vérification de l'overlap entre chunks consécutifs
if len(article_chunks) >= 2:
    c1 = article_chunks[0].page_content
    c2 = article_chunks[1].page_content

    # Trouve le texte commun à la fin de c1 et au début de c2
    overlap_text = ''
    for i in range(min(200, len(c1), len(c2)), 0, -1):
        if c1[-i:] == c2[:i]:
            overlap_text = c1[-i:]
            break

    print('── Vérification de l\'overlap ──────────────────────')
    print(f'Fin chunk 1   : ...{c1[-100:]!r}')
    print(f'Début chunk 2 : {c2[:100]!r}...')
    print(f'Overlap détecté : {len(overlap_text)} chars')
    if overlap_text:
        print(f'Texte en commun : {overlap_text!r}')

## 10. Sauvegarde des chunks

In [ ]:
Path(CHUNKS_PATH).parent.mkdir(parents=True, exist_ok=True)

with open(CHUNKS_PATH, 'w', encoding='utf-8') as f:
    for chunk in all_chunks:
        record = {
            'page_content': chunk.page_content,
            'metadata':     chunk.metadata,
        }
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f'Chunks sauvegardés → {CHUNKS_PATH}')
print(f'Total : {len(all_chunks)} chunks')

# Vérification du rechargement
with open(CHUNKS_PATH) as f:
    count = sum(1 for _ in f)
print(f'Vérification rechargement : {count} lignes ✓')

## 11. Conclusions et décisions pour l'indexing

### Résultats du chunking

| Métrique | Valeur |
|---|---|
| Articles en entrée | 13 644 |
| Chunks en sortie | voir ci-dessus |
| Ratio d'expansion | ~1.5–2x |
| Longueur moyenne chunk | ~350–450 chars |

### Décisions techniques clés

1. **RecursiveCharacterTextSplitter** plutôt que fixed-size : respecte la ponctuation juridique (`.`, `;`) — les articles de loi sont structurés en alinéas numérotés

2. **Stratégie adaptative** selon la longueur : 60% des articles ne sont pas découpés du tout — découper des articles de 200 chars en chunks de 512 serait contre-productif

3. **parent_content embarqué** : prépare le parent-child retrieval — en phase 3, on pourra retriever sur les petits chunks et envoyer l'article complet au LLM

4. **chunk_id unique** : `{article_id}_{index}` — permet la déduplication et le debugging

### Prochaine étape — Phase 3 : Indexing

```python
# Ce qu'on va faire en phase 3
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings  # ou sentence-transformers

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory='data/index/chroma'
)
```